# **Game Review Sentiment Analysis and Semantic Textual Similarity Pipeline**

This notebook implements a comprehensive pipeline for:
1. Sentiment analysis using multiple techniques (TextBlob, VADER, NLTK, BERT)
2. Review aggregation and preprocessing
3. Semantic similarity-based text summarization
4. Evaluation using ROUGE, BLEU, METEOR, and BERTScore metrics

**After this you can easily implement summarization models on Reviews.**

Author: Ubaid Raza

Date: 25 September 2025


# SECTION 1: Environment Setup (Library Installation & Importing Required Librarires

In [ ]:
# Download required NLTK data
!python -m textblob.download_corpora
!python -m spacy download en_core_web_lg

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('state_union')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('vader_lexicon')
nltk.download('movie_reviews')
warnings.filterwarnings('ignore')

import nltk
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score
from textblob import TextBlob
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import warnings
import random



# SECTION 2: Data Loading and Exploration

In [ ]:
# Load data
df = pd.read_excel('/content/Sample_Dataset_of_games.xlsx')


# Display dataset overview
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nDataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")

# Remove null reviews
initial_count = len(df)
df = df.dropna(subset=['Review'])
removed_count = initial_count - len(df)
print(f"\nRemoved {removed_count} rows with null reviews")

# Genre distribution
if 'Genre' in df.columns and 'Game Name' in df.columns:
    genre_stats = []
    for genre in df['Genre'].unique():
        genre_data = df[df['Genre'] == genre]
        genre_stats.append({
            'Genre': genre,
            'Unique Games': genre_data['Game Name'].nunique(),
            'Total Reviews': len(genre_data)
        })
    genre_df = pd.DataFrame(genre_stats)
    print(f"\n{genre_df.to_string(index=False)}")




# SECTION 3: Sentiment Analysis Implementations

## Technique 1: TextBlob

In [ ]:
def get_sentiment_category_textblob(review):
    if isinstance(review, str):
        analysis = TextBlob(review)
        polarity = analysis.sentiment.polarity
        if polarity > 0:
            return 'positive'
        elif polarity < 0:
            return 'negative'
        else:
            return 'neutral'
    else:
        return 'undefined'

df['Sentiment_TextBlob'] = df['Review'].apply(get_sentiment_category_textblob)
textblob = df[['Game Name', 'Review', 'Sentiment_TextBlob']].copy()

## Technique 2: Vader

In [ ]:
def get_sentiment_category_vader(review):
    review = str(review)
    sia = SentimentIntensityAnalyzer()
    compound_score = sia.polarity_scores(review)['compound']
    if compound_score >= 0.05:
        return 'positive'
    elif compound_score <= -0.05:
        return 'negative'
    else:
        return 'neutral'
df['Sentiment_VADER'] = df['Review'].apply(get_sentiment_category_vader)
sentiment_vader = df[['Game Name', 'Review', 'Sentiment_VADER']].copy()



## Technique 3: Bert

In [ ]:
# Load BERT model and tokenizer once
bert_tokenizer = AutoTokenizer.from_pretrained("juliensimon/reviews-sentiment-analysis")
bert_model = AutoModelForSequenceClassification.from_pretrained("juliensimon/reviews-sentiment-analysis")

def analyze_sentiment_bert(review):
    if isinstance(review, str):
        review = [review]
    inputs = bert_tokenizer(review, return_tensors="pt", truncation=True, padding=True, max_length=512)
    outputs = bert_model(**inputs)
    logits = outputs.logits
    predicted_classes = logits.argmax(dim=1)
    sentiments = []
    for predicted_class in predicted_classes:
        if predicted_class == 0:
            sentiments.append('negative')
        elif predicted_class == 1:
            sentiments.append('positive')
        else:
            sentiments.append('neutral')
    return sentiments[0] if len(sentiments) == 1 else sentiments


df['Sentiment_BERT'] = df['Review'].apply(analyze_sentiment_bert)
sentiment_bert = df[['Game Name', 'Review', 'Sentiment_BERT']].copy()

# SECTION 4: Sentiment Analysis Evaluation


In [ ]:
# Print counts for TextBlob
print("Print counts for TextBlob")
print("Negative values is: ", len(textblob[textblob["Sentiment_TextBlob"] == "negative"]))
print("positive values is: ", len(textblob[textblob["Sentiment_TextBlob"] == "positive"]))
print("Neutral values is: ", len(textblob[textblob["Sentiment_TextBlob"] == "neutral"]))

# Print counts for VADER
print("Print counts for VADER")
print("Negative values is: ", len(sentiment_vader[sentiment_vader["Sentiment_VADER"] == "negative"]))
print("positive values is: ", len(sentiment_vader[sentiment_vader["Sentiment_VADER"] == "positive"]))
print("Neutral values is: ", len(sentiment_vader[sentiment_vader["Sentiment_VADER"] == "neutral"]))

# Print counts for BERT
print("Print counts for Bert")
print("Negative values is: ", len(sentiment_bert[sentiment_bert["Sentiment_BERT"] == "negative"]))
print("Positive values is: ", len(sentiment_bert[sentiment_bert["Sentiment_BERT"] == "positive"]))
print("Neutral values is: ", len(sentiment_bert[sentiment_bert["Sentiment_BERT"] == "neutral"]))

#Calculating Auc Scores
# Technique 1: TextBlob
positive_mask_textblob = textblob['Sentiment_TextBlob'] == 'positive'
negative_mask_textblob = textblob['Sentiment_TextBlob'] == 'negative'
positive_scores_textblob = textblob.loc[positive_mask_textblob, 'Sentiment_TextBlob'].index.values + 1
negative_scores_textblob = textblob.loc[negative_mask_textblob, 'Sentiment_TextBlob'].index.values + 1
auc_textblob = (sum(positive_scores_textblob) - (len(positive_scores_textblob) * (len(positive_scores_textblob) + 1) / 2)) / (len(positive_scores_textblob) * len(negative_scores_textblob))
print(f'AUC for TextBlob: {auc_textblob}')

# Technique 2: VADER
positive_mask_vader = sentiment_vader['Sentiment_VADER'] == 'positive'
negative_mask_vader = sentiment_vader['Sentiment_VADER'] == 'negative'
positive_scores_vader = sentiment_vader.loc[positive_mask_vader, 'Sentiment_VADER'].index.values + 1
negative_scores_vader = sentiment_vader.loc[negative_mask_vader, 'Sentiment_VADER'].index.values + 1
auc_vader = (sum(positive_scores_vader) - (len(positive_scores_vader) * (len(positive_scores_vader) + 1) / 2)) / (len(positive_scores_vader) * len(negative_scores_vader))


# Technique 3: BERT Sentiment
positive_mask_bertsentiment = sentiment_bert['Sentiment_BERT'] == 'positive'
negative_mask_bertsentiment = sentiment_bert['Sentiment_BERT'] == 'negative'
positive_scores_bertsentiment = sentiment_bert.loc[positive_mask_bertsentiment, 'Sentiment_BERT'].index.values + 1
negative_scores_bertsentiment = sentiment_bert.loc[negative_mask_bertsentiment, 'Sentiment_BERT'].index.values + 1
auc_bert = (sum(positive_scores_bertsentiment) - (len(positive_scores_bertsentiment) * (len(positive_scores_bertsentiment) + 1) / 2)) / (len(positive_scores_bertsentiment) * len(negative_scores_bertsentiment))


print(f'AUC for TextBlob: {auc_textblob}')
print(f'AUC for VADER: {auc_vader}')
print(f'AUC for Bertsentiment: {auc_bert}')

# Section 5: Combining Negative Reviews Of Unique Game By Using Best Sentiment Analysis Technique

In [ ]:
# Combining Negative Reviews Of Unique Game By Using VADER
df_combined = sentiment_vader
name = []
summar = []
for game_name in df_combined['Game Name'].unique():
    game_reviews = df_combined[(df_combined['Game Name'] == game_name) & (df_combined['Sentiment_VADER'] == 'negative')]['Review'].tolist()
    if game_reviews:
        name.append(game_name)
        summar.append(' '.join(game_reviews))
gamedata = pd.DataFrame({"Game Name": name, "Negative_Reviews": summar})

def count_words(review):
    words = review.split()
    return len(words)

gamedata['Word Count'] = gamedata['Negative_Reviews'].apply(count_words)
gamedata.to_excel('/content/sentiment_summaries.xlsx', index=False)

# Section 6: Text Preprocessing on Negative Reviews



In [ ]:
# Text Preprocessing
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s.]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s.]', '', text)
    text = re.sub(r'(?<!\.)\s+', ' ', text)
    word_tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    word_tokens = [word for word in word_tokens if word not in stop_words]
    ps = PorterStemmer()
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(ps.stem(word)) for word in word_tokens]
    text = ' '.join(lemmatized_tokens)
    return text

text_df = pd.DataFrame({
    "Game Name": gamedata["Game Name"],
    "Negative_Reviews": gamedata["Negative_Reviews"].apply(preprocess_text)
})

text_df.to_excel('textpreprocessing_summaries.xlsx', index=False)

# Section 7: Semantic Textual Similarity on Reviews

## Technique 1: Sentence Transformer

In [ ]:
input_file = '/content/textpreprocessing_summaries.xlsx'
df_input = pd.read_excel(input_file)
model_st = SentenceTransformer("Sakil/sentence_similarity_semantic_search")

def create_modified_review_st(review, model):
    sentences = [sent.strip() for sent in review.split('.') if sent.strip()]
    embeddings = model.encode(sentences, convert_to_tensor=True)
    threshold = 0.7
    modified_sentences = []
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            similarity = util.pytorch_cos_sim(embeddings[i], embeddings[j])[0][0].item()
            if similarity >= threshold:
                modified_sentences = sentences[:i + 1] + sentences[j:]
                break
    if not modified_sentences:
        modified_sentences = sentences
    modified_review = ' '.join([sent + '.' for sent in modified_sentences])
    return modified_review if modified_review.strip() else review

summary_df_st = pd.DataFrame(columns=['Game Name', 'Reduced Summary', 'Word Count'])
for index, row in df_input.iterrows():
    game_name = row['Game Name']
    review = row['Negative_Reviews']
    if pd.notnull(review):
        reduced_summary = create_modified_review_st(review, model_st)
        token_count = len(model_st.tokenizer.encode(reduced_summary, add_special_tokens=False))  # Adjusted for model tokenizer
        new_row = pd.DataFrame([{'Game Name': game_name, 'Reduced Summary': reduced_summary, 'Word Count': token_count}])
        summary_df_st = pd.concat([summary_df_st, new_row], ignore_index=True)

output_file_st = 'sentencetransformer.xlsx'
summary_df_st.to_excel(output_file_st, index=False)

## Technique 2: SBERT

In [ ]:
model_sbert = SentenceTransformer("distilbert-base-nli-mean-tokens")

def create_modified_review_sbert(review, model):
    sentences = [sent.strip() for sent in review.split('.') if sent.strip()]
    embeddings = model.encode(sentences, convert_to_tensor=True)
    threshold = 0.7
    overlapping_sentences = []
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            similarity = util.pytorch_cos_sim(embeddings[i], embeddings[j])[0][0].item()
            if similarity >= threshold:
                overlapping_sentences.append(sentences[i] + '.')
                break
    modified_review = ' '.join(overlapping_sentences)
    if modified_review and modified_review[-1] != '.':
        modified_review += '.'
    return modified_review

df_sbert = df_input.copy()
df_sbert['summary'] = df_sbert['Negative_Reviews'].apply(lambda r: create_modified_review_sbert(r, model_sbert))
output_df_sbert = df_sbert[['Game Name', 'Negative_Reviews', 'summary']]
output_df_sbert.to_excel('/content/sbert.xlsx', index=False)

## Technique 3: SEBERT

In [ ]:
model_sebert = SentenceTransformer("paraphrase-MiniLM-L6-v2")

def create_modified_review_sebert(review, model):
    sentences = [sent.strip() for sent in review.split('.') if sent.strip()]
    embeddings = model.encode(sentences, convert_to_tensor=True)
    threshold = 0.7
    overlapping_sentences = []
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            similarity = util.pytorch_cos_sim(embeddings[i], embeddings[j])[0][0].item()
            if similarity >= threshold:
                overlapping_sentences.append(sentences[i] + '.')
                break
    return ' '.join(overlapping_sentences)

df_sebert = df_input.copy()
df_sebert['summary'] = df_sebert['Negative_Reviews'].apply(lambda r: create_modified_review_sebert(r, model_sebert))
output_df_sebert = pd.DataFrame({
    'Game Name': df_sebert['Game Name'],
    'Original Reviews': df_sebert['Negative_Reviews'],
    'Output Reviews': df_sebert['summary']
})
output_df_sebert.to_excel('/content/sebert.xlsx', index=False)

# Section 8: Evaluation of Semantic Textual Similarity

In [ ]:
def evaluate_summaries(original_df, reduced_df, original_col='Negative_Reviews', reduced_col='Reduced Summary'):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    smoothing = SmoothingFunction()
    rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
    bleu_scores, meteor_scores, bert_scores = [], [], []
    precision_scores, recall_scores = [], []

    for _, (orig_row, red_row) in enumerate(zip(original_df.iterrows(), reduced_df.iterrows())):
        original_review = orig_row[1][original_col]
        reduced_summary = red_row[1][reduced_col]
        if pd.notnull(original_review) and pd.notnull(reduced_summary):
            rouge_scores = scorer.score(reduced_summary, original_review)
            rouge1_scores.append(rouge_scores['rouge1'].fmeasure)
            rouge2_scores.append(rouge_scores['rouge2'].fmeasure)
            rougeL_scores.append(rouge_scores['rougeL'].fmeasure)

            bleu = sentence_bleu([original_review.split()], reduced_summary.split(), smoothing_function=smoothing.method1)
            bleu_scores.append(bleu)

            tokenized_original = original_review.split()
            tokenized_reduced = reduced_summary.split()
            meteor = single_meteor_score(tokenized_original, tokenized_reduced)
            meteor_scores.append(meteor)

            _, _, bert_f1 = bert_score([reduced_summary], [original_review], lang='en', model_type='bert-base-uncased')
            bert_scores.append(bert_f1.item())

            human_set = set(tokenized_original)
            reduced_set = set(tokenized_reduced)
            precision = len(human_set.intersection(reduced_set)) / len(reduced_set) if len(reduced_set) > 0 else 0
            recall = len(human_set.intersection(reduced_set)) / len(human_set) if len(human_set) > 0 else 0
            precision_scores.append(precision)
            recall_scores.append(recall)

    corpus_bleu_score = corpus_bleu(
        [[orig.split()] for orig in original_df[original_col]],
        [red.split() for red in reduced_df[reduced_col]],
        smoothing_function=smoothing.method1
    )

    print("ROUGE-1 F1 Score:", sum(rouge1_scores) / len(rouge1_scores))
    print("ROUGE-2 F1 Score:", sum(rouge2_scores) / len(rouge2_scores))
    print("ROUGE-L F1 Score:", sum(rougeL_scores) / len(rougeL_scores))
    print("Average BLEU Score (Sentence-Level):", sum(bleu_scores) / len(bleu_scores))
    print("Corpus BLEU Score (Corpus-Level):", corpus_bleu_score)
    print("Average METEOR Score:", sum(meteor_scores) / len(meteor_scores))
    print("Average BERTScore:", sum(bert_scores) / len(bert_scores))
    print("Average Precision:", sum(precision_scores) / len(precision_scores))
    print("Average Recall:", sum(recall_scores) / len(recall_scores))

# Call evaluations for each method

print("Evaluation for Sentence Transformer:")
evaluate_summaries(df_input, summary_df_st, reduced_col='Reduced Summary')

print("Evaluation for Sbert")
evaluate_summaries(df_input, output_df_sbert, reduced_col='summary')

print("Evaluation for Sebert")
evaluate_summaries(df_input, output_df_sebert, original_col='Negative_Reviews', reduced_col='Output Reviews')

Evaluation for Sentence Transformer:
ROUGE-1 F1 Score: 0.9861813763825341
ROUGE-2 F1 Score: 0.9857122401795142
ROUGE-L F1 Score: 0.9861813763825341
Average BLEU Score (Sentence-Level): 0.6567598280697564
Corpus BLEU Score (Corpus-Level): 0.6651442106321435
Average METEOR Score: 0.7584990120290505
Average BERTScore: 0.9853694622333233
Average Precision: 0.8374370142885813
Average Recall: 0.9058114836478478
Evaluation for Sbert
ROUGE-1 F1 Score: 0.8347058490142315
ROUGE-2 F1 Score: 0.8169272143295767
ROUGE-L F1 Score: 0.8347058490142315
Average BLEU Score (Sentence-Level): 0.4512490810137365
Corpus BLEU Score (Corpus-Level): 0.48591748307865046
Average METEOR Score: 0.4816807070819846
Average BERTScore: 0.9243722191223731
Average Precision: 0.8467864329039032
Average Recall: 0.7796060734638672
Evaluation for Sebert


ROUGE-1 F1 Score: 0.29803331849750764
ROUGE-2 F1 Score: 0.2943220647096126
ROUGE-L F1 Score: 0.29803331849750764
Average BLEU Score (Sentence-Level): 0.10755032003133198
Corpus BLEU Score (Corpus-Level): 0.04769661501811221
Average METEOR Score: 0.17694743656017173
Average BERTScore: 0.6709678218914912
Average Precision: 0.6984626687305936
Average Recall: 0.3771820663167256


# Section 9: Selection the Best STS Technique

In [ ]:
final_df = pd.DataFrame({
    "Game Name": summary_df_st["Game Name"],
    'Neg_Review_Summary': summary_df_st["Reduced Summary"]
})
final_df.to_excel('/content/summariesofgames.xlsx', index=False)